[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoAvancadoIME/blob/main/01_revisao_ajustamento_basico.ipynb)

# Aula 1 — Revisão de Ajustamento Básico

**Maj Diego - 1º Semestre / 2027**

**Objetivos:**

1. Revisar Cálculo vetorial
2. Revisar o Método dos Mínimos Quadrados (MMQ) modelagem geral.
3. Avaliar a qualidade do ajustamento por resíduos, variâncias e testes estatísticos.

## 1. Revisão Cálculo vetorial

### **1.1 Gradiente de uma função escalar com variável vetorial**

**Definição 1 (Gradiente).** Dada uma função escalar de variável vetorial $f(\mathbf{x})$, o
gradiente de $f$ é

$$
\nabla f(\mathbf{x}) =
\begin{bmatrix}
\dfrac{\partial f}{\partial x_1}\\[4pt]
\dfrac{\partial f}{\partial x_2}\\[2pt]
\vdots\\[2pt]
\dfrac{\partial f}{\partial x_n}
\end{bmatrix}
$$

O operador $\nabla f(\mathbf{x})$ possui uma interpretação geométrica, a resultante aponta na direção de crescimento mais rápido de $f$ a partir do ponto considerado. Quando $\nabla f(\mathbf{\hat{x}}) = \mathbf{0}$, $\mathbf{\hat{x}}$
é chamado **ponto crítico**.

### **1.2 Gradiente de uma função vetorial com variável vetorial**

**Definição 2 (Jacobiana).** Para uma função vetorial
$F(\mathbf{x}) = [f_1(\mathbf{x}), \dots, f_m(\mathbf{x})]^T$, a Jacobiana é

$$
J(\mathbf{x}) =
\begin{bmatrix}
\dfrac{\partial f_1}{\partial x_1} & \cdots & \dfrac{\partial f_1}{\partial x_n}\\
\vdots & \ddots & \vdots\\
\dfrac{\partial f_m}{\partial x_1} & \cdots & \dfrac{\partial f_m}{\partial x_n}
\end{bmatrix}
$$

As linhas de $J(\mathbf{x})$ são os gradientes (transpostos) de $f_1, \dots, f_m$. Alguns
autores escrevem $\nabla F(\mathbf{x})$ para a Jacobiana.

**Exercício 1**: Tirando o verniz

Calcule $\nabla f(\mathbf{x})$ para:

a) $f(\mathbf{x}) = \mathbf{c}^T\mathbf{x}$, com $\mathbf{c}$ constante

b) $f(\mathbf{x}) = \mathbf{x}^T\mathbf{x}$

c) $f(\mathbf{x}) = \mathbf{x}^T A \mathbf{x}$

d) $f(\mathbf{x}) = \lVert A\mathbf{x} - \mathbf{b}\rVert^2$

**Solução exercício 1 (comentada).** 

<!-- Seja $\mathbf{u}_j$ o vetor canônico (zeros exceto 1 na posição $j$).

**a)** $\dfrac{\partial \mathbf{c}^T\mathbf{x}}{\partial x_j} = c_j = \mathbf{u}_j^T\mathbf{c}$,
logo $\nabla f(\mathbf{x}) = I\mathbf{c} = \mathbf{c}$.

**b)** $\dfrac{\partial \mathbf{x}^T\mathbf{x}}{\partial x_j} = 2\mathbf{u}_j^T\mathbf{x}$,
logo $\nabla f(\mathbf{x}) = 2\mathbf{x}$.

**c)** $\dfrac{\partial \mathbf{x}^T A\mathbf{x}}{\partial x_j} =
\dfrac{\partial \mathbf{x}^T}{\partial x_j}A\mathbf{x} + \mathbf{x}^T\dfrac{\partial A\mathbf{x}}{\partial x_j}
= 2\mathbf{u}_j^T A\mathbf{x}$ (usando $A$ simétrica), logo $\nabla f(\mathbf{x}) = 2A\mathbf{x}$.

**d)** Expandindo,
$f(\mathbf{x}) = \mathbf{x}^TA^TA\mathbf{x} - 2\mathbf{b}^TA\mathbf{x} + \mathbf{b}^T\mathbf{b}$,
logo $\nabla f(\mathbf{x}) = 2A^TA\mathbf{x} - 2A^T\mathbf{b}$. 

Verificamos simbolicamente abaixo com `sympy`:



```python
import numpy as np
import sympy as sp
from scipy.optimize import least_squares
np.set_printoptions(precision=6, suppress=True)
n = 3
x = sp.Matrix(sp.symbols(f'x1:{n+1}'))
c = sp.Matrix(sp.symbols(f'c1:{n+1}'))
A = sp.Matrix(n, n, lambda i, j: sp.Symbol(f'a{i+1}{j+1}'))
A = (A + A.T) / 2          # força simetria, como assumido na dedução de (c)
b = sp.Matrix(sp.symbols(f'b1:{n+1}'))

def grad(f, x):
    return sp.Matrix([sp.diff(f, xi) for xi in x])

# a) c^T x
fa = (c.T * x)[0]
ga = grad(fa, x)
print("a) grad(c^T x) - c :", sp.simplify(ga - c))

# b) x^T x
fb = (x.T * x)[0]
gb = grad(fb, x)
print("b) grad(x^T x) - 2x:", sp.simplify(gb - 2*x))

# c) x^T A x
fc = (x.T * A * x)[0]
gc = grad(fc, x)
print("c) grad(x^T A x) - 2Ax:", sp.simplify(gc - 2*A*x))

# d) ||Ax - b||^2
fd = ((A*x - b).T * (A*x - b))[0]
gd = grad(fd, x)
print("d) grad(||Ax-b||^2) - (2A^TAx - 2A^Tb):", sp.simplify(gd - (2*A.T*A*x - 2*A.T*b)))

```
-->

## 2. Método dos Mínimos Quadrados

O MMQ estima os parâmetros que minimizam uma função objeto $\Phi=V^TPV$, em que $P \in \mathbb{R}^{n \times n}$ é a matriz dos pesos e $V \in \mathbb{R}^n$ é o vetor dos resíduos, função dos parâmetros $X$, assim $\Phi(X)$ e a solução passa por:

$$ \frac{\partial \Phi}{\partial X} = 0$$
Extende-se o quanto necessário (com mais variáveis) a idéia acima: $\Phi=V^TPV + K^T(AX + BV + W)$, em que $K$ é o vetor de correlatos, $A$ e $B$ são as matrizes de coeficientes dos parâmetros e dos resíduos, respectivamente, assim $\Phi(X,V,K)$ e a solução passa por:

$$ \frac{\partial \Phi}{\partial X} = 0 \ , \ \frac{\partial \Phi}{\partial K} = 0 \ \ e \ \ \frac{\partial \Phi}{\partial V} = 0 $$

### **2.1 Teorema de Taylor**

**Teorema 1.** Se $f(\mathbf{x})$ e suas derivadas primeira e segunda são contínuas, existe $\mathbf{c}$ entre $\mathbf{x}$ e $\mathbf{x}+\Delta\mathbf{x}$ tal que

$$
f(\mathbf{x}+\Delta\mathbf{x}) = f(\mathbf{c}) + \nabla f(\mathbf{c})^T\Delta\mathbf{x}
+ \tfrac12 \Delta\mathbf{x}^T \nabla^2 f(\mathbf{c})\,\Delta\mathbf{x}
$$

Para $\Delta\mathbf{x}$ pequeno,

$$
f(\mathbf{x}+\Delta\mathbf{x}) \approx f(\mathbf{x}) + \nabla f(\mathbf{x})^T\Delta\mathbf{x}
+ \tfrac12 \Delta\mathbf{x}^T \nabla^2 f(\mathbf{x})\,\Delta\mathbf{x}
$$

e a **aproximação linear** (só com derivadas primeiras) é
$f(\mathbf{x}+\Delta\mathbf{x}) \approx f(\mathbf{x}) + \nabla f(\mathbf{x})^T\Delta\mathbf{x}$.

Para uma função vetorial $F$, a versão linearizada é

$$
F(\mathbf{x}+\Delta\mathbf{x}) \approx F(\mathbf{x}) + J(\mathbf{x})\,\Delta\mathbf{x}
$$

Esta é a base de todo método de linearização usado pelo MMQ.

### **2.2 Multiplicadores de Lagrange**

O método resolve problemas de otimização restrita.

**Teorema 2.** O problema 

$$\begin{cases}\min f(\mathbf{x}) \\ g(\mathbf{x})=0 \end{cases}$$

só pode ocorrer em $\mathbf{\hat{x}}$ tal que $\nabla f(\mathbf{\hat{x}}) = \lambda \nabla g(\mathbf{\hat{x}})$, para
algum $\lambda$. 

O que nos permite escrever uma nova função, chamada **função Lagrangiana** $\mathcal{L}$:

$$\nabla_\mathbf{\hat{x}} \mathcal{L}(\mathbf{\hat{x}}, \lambda) = \nabla f(\mathbf{\hat{x}}) - \lambda \nabla g(\mathbf{\hat{x}}) = 0$$
$$\mathcal{L}(\mathbf{\hat{x}}, \lambda) = f(\mathbf{\hat{x}}) - \lambda g(\mathbf{\hat{x}})$$

Agora, em vez de procurar apenas $\mathbf{\hat{x}}$, procuramos simultaneamente, $\mathbf{\hat{x}}$ e $\lambda$ que minimiza a Lagrangiana:

$$\argmin_{\mathbf{\hat{x}}, \lambda} \mathcal{L}(\mathbf{\hat{x}}, \lambda) \rightarrow \nabla \mathcal{L}=0 \rightarrow \nabla_{\mathbf{\hat{x}}} \mathcal{L}=0 \ \ \text{ e } \ \nabla_{\lambda} \mathcal{L}=0$$

### **2.3 Método de Lagrange aplicado na modelagem geral**

$$\begin{cases}\underset{X}{\operatorname{argmin }} (\Phi = V^TPV) \\ AX + BV + W =0 \end{cases}$$


$$ \underset{V,K,X}{\operatorname{argmin}} (V^TPV) \qquad\text{ sujeito a }\qquad AX + BV + W =0 $$

$$\Psi = V^TPV - 2K^T(AX + BV + W)$$

$$ \underset{V,K,X}{\operatorname{argmin}}(\Psi) \rightarrow \frac{\partial {\Psi}}{\partial{V}} = 0 \ \ , \frac{\partial {\Psi}}{\partial{K}}=0 \ \ \text{ e } \ \  \frac{\partial {\Psi}}{\partial{X}}  = 0 $$

$$\text{Hipermatriz: }\begin{bmatrix}P & B^T & 0\\ B & 0 & A \\ 0 & A^T & 0 \end{bmatrix}\begin{bmatrix} V \\ K \\ X \end{bmatrix} + \begin{bmatrix} 0 \\ W \\0 \end{bmatrix} = 0$$

$$\text{Solução: }\begin{bmatrix} V \\ K \\ X \end{bmatrix} = -\begin{bmatrix}P & B^T & 0\\ B & 0 & A \\ 0 & A^T & 0 \end{bmatrix}^{-1} \begin{bmatrix} 0 \\ W \\0 \end{bmatrix} = 0$$



**Exercício 02: Método de Helmert (1872)**

Montar o sistema matricial para a solução de:
$$
\min \tfrac12\lVert AX-\mathbf{b}\rVert^2 \quad \text{sujeito a} \quad CX=\mathbf{d}
$$

**Solução exercício 02 (comentada)** 

<!-- Com o lagrangiano
$\mathcal{L}(X,K) = \tfrac12(AX-\mathbf{b})^T(AX-\mathbf{b}) + K^T(CX-\mathbf{d})$,

$$
\frac{\partial\mathcal{L}}{\partial X} = A^TAX - A^T\mathbf{b} + C^TK = 0
\;\Rightarrow\; A^TAX + C^TK = A^T\mathbf{b}
$$
$$
\frac{\partial\mathcal{L}}{\partial K} = CX-\mathbf{d} = 0
$$

logo o sistema normal do problema é

$$
\begin{bmatrix} A^TA & C^T \\ C & 0 \end{bmatrix}
\begin{bmatrix} X \\ K \end{bmatrix}
=
\begin{bmatrix} A^T\mathbf{b} \\ \mathbf{d} \end{bmatrix}
$$ -->

## 3. Qualidade pós-ajustamento

Após obter a solução ajustada, é necessário verificar se ela é **estatisticamente compatível** com as observações e com o modelo estocástico adotado. Resíduos pequenos, isoladamente, não garantem um bom ajustamento: eles devem ser avaliados em relação às suas precisões, à redundância do sistema e ao fator de variância esperado.

### **3.1 Graus de liberdade**

Os graus de liberdade representam a redundância global disponível para avaliar o ajustamento:

$$
gl=m-n.
$$

É necessário que $gl>0$ para estimar o fator de variância a posteriori. Quando $gl=0$, o sistema pode ser resolvido, mas não há redundância para uma avaliação estatística interna.

### **3.2 Fator de variância a posteriori**

O fator de variância a posteriori é estimado por

$$
\hat\sigma_0^2=\frac{V^TPV}{gl}.
$$

Se a matriz de pesos foi construída com um fator de variância a priori $\sigma_0^2$, compara-se $\hat\sigma_0^2$ com esse valor. Uma diferença relevante pode indicar pesos mal escalados, erros grosseiros, modelo funcional incompleto ou hipóteses estocásticas inadequadas. Essa comparação deve ser formalizada pelo teste global, e não apenas pela proximidade numérica entre os dois fatores.

### **3.3 Precisão dos parâmetros e das observações ajustadas**

Os problemas de ajustamento surgem com variâncias das observações conhecidas ou extraídas das especificações do manual do instrumento de medida. Assim temos $\Sigma_L$, a propagação das variâncias até os parâmetros e observações ajustadas $X_a$ e $L_a$ é:

MVC dos parâmetros ajustados:

$$
X_a = D_x(L) 
$$

$$\text{Lei de propagação das variâncias } \rightarrow \Sigma_{X_a} = \frac{\partial D_x}{\partial X}\Sigma_L \frac{\partial D_x}{\partial X}^T
$$

e a MVC das observações ajustadas:

$$
L_a = D_l(L) 
$$

$$\text{Lei de propagação das variâncias } \rightarrow \Sigma_{L_a} = \frac{\partial D_l}{\partial X}\Sigma_L \frac{\partial D_l}{\partial X}^T
$$

### **3.4 Teste global do modelo**

Quando $\sigma_0^2$ é conhecido a priori e os erros são normais, a estatística

$$
\chi_{calc} = \frac{\hat{\sigma}_0^2}{\sigma_0^2} gl = \frac{V^TPV}{\sigma_0^2} = V^T\Sigma_L^{-1}V
$$

segue uma distribuição qui-quadrado com $gl$ graus de liberdade. Para um nível de significância $\alpha$, aceita-se a hipótese de compatibilidade global
$H_0:\hat\sigma_0^2=\sigma_0^2$ quando

$$
\chi^2_{\alpha/2,gl}\leq \chi_{calc}\leq\chi^2_{1-\alpha/2,gl}.
$$

A rejeição de $H_0$ informa uma incompatibilidade global, mas não identifica sozinha qual observação ou qual parte do modelo é responsável pelo problema.

### **3.5 Análise dos resíduos**

A não aceitação pode ser usado como diagnóstico inicial, onde os resíduos encontrados são incompatíveis com as variâncias inicialmente estipuladas.

Dentre os sinais de alerta, o mais óbvio é de que exista um *outlier* nas observações, mas uma observação não deve ser eliminada automaticamente apenas por ultrapassar um limiar. Devem ser consideradas diversos outras opções:

- aumentar o grau de liberdade do sistema ($gl>30$) para se encaixar na distribuição $\chi^2$;
- considerar aumentar o nível de significância;
- testar outros modelos estocásticos;
- considerar contexto físico da observação;
- padrões sistemáticos que não forma considerados;
- correlações não representadas no modelo estocástico.

## Laboratório nº 1

Faça uma análise crítica do <a href="media/texts/paper.md">artigo anexado</a>. Espera-se que sua análise tenha entre 2 e 3 páginas.

Embora não seja necessário reajustar a rede, espera-se que, caso você considere que o autor esteja equivocado, apresente as razões para isso e explique como realizaria corretamente a tarefa.

A seguir, apresenta-se uma lista de tópicos que você poderá discutir. Entretanto, você não está limitado apenas a esses tópicos, nem precisa abordar todos eles caso considere desnecessário.

### Possíveis tópicos a serem discutidos

1. Situação geral ou problema;
2. Modelo matemático;
3. Método de ajustamento;
4. Graus de liberdade;
5. Precisão;
6. Acurácia;
7. Sistema de coordenadas planas;
8. Pesos;
9. Entre outros.
